# 06 · 스트림과 비동기 처리

> **CuPy 2일 집중 코스 — Day 1 / 단원 4 (스트림과 비동기 처리) — Day 1 마무리**

단원 3(메모리)과 한 묶음으로, "데이터 이동을 어떻게 **숨기나(overlap)**"가 주제입니다.
스트림·이벤트로 연산과 전송을 겹치고, 이중 버퍼 파이프라인·CUDA Graph까지 다룹니다.

### 왜 스트림을 배우는가

`05_memory_profiling`에서 확인했듯, 많은 GPU 워크로드는 연산 자체가 아니라 **host↔device 전송**에
발목이 잡힙니다(PCIe 대역폭은 GPU 내부 메모리 대역폭보다 한 자릿수 이상 느립니다). 지금까지는
"전송을 줄이자"가 처방이었다면, 이 노트북은 한 걸음 더 나아가 "**줄일 수 없는 전송이라면 연산
뒤에 숨기자**"는 전략을 다룹니다. GPU에는 연산을 담당하는 **SM(Streaming Multiprocessor)** 과
데이터 이동을 담당하는 **복사 엔진(copy engine, DMA)** 이 물리적으로 분리되어 있어, 조건만 맞으면
둘이 **동시에** 일할 수 있습니다. 이 동시성을 프로그램 차원에서 표현하는 도구가 **스트림(stream)**
이고, 스트림 간 순서 의존성을 조율하는 도구가 **이벤트(event)** 입니다.

### 이 노트북의 흐름

개념(스트림·이벤트) → 다중 스트림 의존성 표현(`wait_event`) → 비동기 전송의 전제조건(pinned
메모리) → 실전 패턴(이중 버퍼 청크 파이프라인, 직접 구현 연습) → 병목 진단(이벤트 세분 타이밍,
스트림 수 스케일링) → 프로파일러 연계(NVTX + Power Iteration) → (심화) CUDA Graph·멀티-GPU.
3-5절이 이 노트북의 핵심이고, 6-7절은 "실제로 얼마나 겹쳤는지"를 정량적으로 확인하는 도구입니다.

## 학습 목표
- 스트림(순서 실행)과 다른 스트림 간 **오버랩**, 이벤트 동기화를 이해한다.
- **이중 버퍼 청크 파이프라인**으로 전송-연산을 겹치고 속도를 측정한다.
- 이벤트로 구간을 세분 측정하고, 스트림 수를 스케일링한다.
- (심화) **CUDA Graph 캡처**로 런치 오버헤드를 줄이고, 멀티-GPU를 맛본다.

## 목차
1. [스트림이란](#1)
2. [이벤트로 구간 측정](#2)
3. [다중 스트림 + 이벤트 동기화](#3)
4. [비동기 전송 & pinned memory](#4)
5. [이중 버퍼 청크 오버랩 파이프라인](#5)
6. [이벤트 세분 타이밍](#6)
7. [스트림 수 스케일링](#7)
8. [NVTX/프로파일 연계 + Power Iteration](#8)
9. [(심화) CUDA Graph 캡처](#9)
10. [(심화) 멀티-GPU](#10)
11. [체크포인트](#11)

In [ ]:
import os, sys, time, math
import numpy as np
import cupy as cp
from course_utils import print_env, bench, gpu_ms, cpu_ms, print_bench, compare
print_env()

<a id="1"></a>
## 1. 스트림이란

**스트림(stream)** 은 *순서대로 실행되는 디바이스 작업의 사슬*입니다. 같은 스트림은 순서 보장,
**다른 스트림끼리는 겹쳐(overlap)** 실행될 수 있습니다. 지정 안 하면 기본 스트림을 씁니다(`cp.cuda.get_current_stream()`).

### 조금 더 구체적으로

스트림을 **고속도로 차선**에 비유하면 이해가 쉽습니다. 같은 차선(스트림)에 진입한 차(작업)들은
반드시 진입한 순서대로 빠져나가지만, 다른 차선(다른 스트림)의 차들은 서로 순서를 신경 쓰지 않고
동시에 달릴 수 있습니다. GPU 하드웨어 관점에서는 스트림 자체가 연산을 빠르게 만드는 것이 아니라,
**"이 작업들은 서로 독립적이니 겹쳐도 된다"** 는 정보를 CUDA 드라이버/스케줄러에 알려주는
역할을 합니다. 겹칠 수 있는 자원(복사 엔진, 여러 SM의 유휴 슬롯)이 실제로 남아있어야 이득이
발생합니다.

- **기본(default/legacy) 스트림**: 아무 스트림도 지정하지 않으면 CuPy는 이 스트림을 씁니다.
  레거시 기본 스트림은 다른 모든 스트림과 **암묵적으로 동기화**되는 특수한 성격이 있어(각 작업이
  다른 스트림의 이전 작업을 기다렸다가 시작), 여러 스트림을 진짜로 겹치려면 **명시적으로 새
  스트림을 만들어야** 합니다(`cp.cuda.Stream(non_blocking=True)`).
- **`non_blocking=True`**: 새로 만든 스트림이 기본 스트림과 암묵적으로 동기화되지 않도록 하는
  옵션입니다. 이 노트북의 예제들은 겹침 효과를 보기 위해 거의 항상 이 옵션을 켭니다.
- **`with stream:` 블록**: 블록 안에서 실행되는 CuPy 연산이 해당 스트림에 올라갑니다. 블록을
  벗어나도 스트림 위 작업이 끝났다는 보장은 없으므로(비동기), 결과가 필요하면 `stream.synchronize()`
  또는 이벤트로 명시적으로 기다려야 합니다 — `00_intro_env` 6절에서 다룬 "비동기 타이밍의 함정"이
  스트림 단위로 다시 등장하는 셈입니다.

> 💡 스트림 개수를 늘린다고 무조건 빨라지지 않습니다. 실제로 겹칠 **독립적인 작업**과 겹칠 **여유
> 하드웨어 자원**(복사 엔진, SM 유휴분)이 있어야 하며, 7절에서 이 한계를 직접 수치로 확인합니다.

<img src="images/figures/new_stream_concept.png" width="560">

<sub>그림: 여러 스트림에 올라간 작업들이 시간축 위에서 겹쳐 실행되는 개념도 — 같은 스트림 내부는
순서가 유지되고, 스트림 사이는 독립적으로 진행됩니다.</sub>



In [ ]:
A = cp.cuda.Stream(non_blocking=True)
a = cp.random.random(10_000_000, dtype=cp.float32)
with A:
    s = (cp.sin(a)+1).sum()
A.synchronize()
print('A 스트림 결과:', float(s))

<a id="2"></a>
## 2. 이벤트로 구간 측정

이벤트는 스트림 위 시점 표식입니다. 두 이벤트 사이 시간으로 GPU 구간을 정확히 잽니다.

### 조금 더 구체적으로

**이벤트(event)** 는 그 자체로는 아무 연산도 하지 않는, "이 스트림에서 지금까지의 작업이 여기까지
진행됐다"는 **타임스탬프 표식**입니다. `event.record()`를 호출하면 해당 이벤트가 현재 스트림의
작업 큐에 삽입되고, GPU가 그 지점까지 실제로 도달했을 때 시각이 기록됩니다.

- **`cp.cuda.get_elapsed_time(start, end)`**: 두 이벤트 사이의 GPU 시간을 밀리초(ms) 단위로,
  보통 마이크로초 수준의 분해능으로 돌려줍니다. `time.perf_counter()` 같은 CPU 타이머와 달리
  **GPU 하드웨어 시계**를 기준으로 하므로, 커널 런치의 비동기성에 영향을 받지 않습니다
  (`00_intro_env` 6절의 "비동기 타이밍의 함정"에서 CPU 타이머가 왜 부정확한지 다룬 바 있습니다).
- **`ed.synchronize()`가 필요한 이유**: 이벤트를 기록하는 것 자체는 비동기이므로, `get_elapsed_time`을
  호출하기 전에 최소한 종료 이벤트가 실제로 발생했는지 확인해야 합니다. 이벤트를 `synchronize()`하면
  스트림 전체가 아니라 **그 이벤트 시점까지만** 기다리므로, `stream.synchronize()`보다 더 세밀한
  제어가 가능합니다.
- **`course_utils.bench`와의 관계**: 00~05에서 써온 `bench`(`cupyx.profiler.benchmark`)는 내부적으로
  바로 이 CUDA 이벤트 쌍을 커널 앞뒤에 자동으로 심어 GPU 시간을 측정합니다. 이번 절은 그 메커니즘을
  직접 손으로 만들어보는 것이라 생각하면 됩니다.
- **활용**: 6절에서는 이벤트 3개(`e0, e1, e2`)로 "전송 구간"과 "연산 구간"을 나눠 측정해 어느 쪽이
  병목인지 진단합니다 — 이벤트가 단순 타이밍 도구를 넘어 **성능 디버깅 도구**로 쓰이는 예입니다.

In [ ]:
a = cp.random.random(30_000_000, dtype=cp.float32)
st=cp.cuda.Event(); ed=cp.cuda.Event()
st.record(); y=(cp.sin(a)+1).sum(); ed.record(); ed.synchronize()
print('구간:', cp.cuda.get_elapsed_time(st,ed),'ms')

<a id="3"></a>
## 3. 다중 스트림 + 이벤트 동기화

여러 스트림에 독립 작업을 올리면 겹칠 수 있고, 스트림 간 의존성은 **이벤트**(`record`/`wait_event`)로 표현합니다.

### 조금 더 구체적으로

두 스트림에 완전히 독립적인 작업만 있다면 아무 조율 없이도 겹쳐 실행됩니다. 하지만 실제로는
"스트림 A의 결과가 있어야 스트림 B가 시작할 수 있는" 상황이 흔합니다(예: A에서 전처리한 데이터를
B에서 계산에 쓰는 경우). 이때 `stream.synchronize()`로 **CPU가 A 전체를 기다렸다가** B를 시작하면
안전하지만 그만큼 겹침 기회를 잃습니다. 대신 **`event.record(streamA)` → `streamB.wait_event(event)`**
패턴을 쓰면, CPU는 계속 다음 코드를 실행하고(넌블로킹), **GPU 스케줄러 차원에서** "B는 그 이벤트가
발생하기 전까지 자기 큐의 뒷부분 작업을 시작하지 말라"는 의존성만 겁니다.

- **`wait_event`는 CPU를 막지 않습니다**: `stream.synchronize()`는 CPU가 GPU를 기다리는 동기화이지만,
  `wait_event`는 **GPU 내부에서 스트림끼리** 순서를 맞추는 것이라 CPU는 바로 다음 파이썬 줄로
  넘어갑니다. 이 차이가 다중 스트림 파이프라인에서 CPU 오버헤드를 줄이는 핵심입니다.
- **바통 넘기기(relay race) 비유**: `e.record(s1)`은 "내 구간이 여기까지 끝났다"는 바통을 놓는
  행위이고, `s2.wait_event(e)`는 "그 바통을 받을 때까지는 출발하지 않는다"는 규칙입니다. s1의 나머지
  작업과 s2가 바통을 기다리는 동안에도 서로 다른 하드웨어 자원(예: s1은 계산 중, s2는 대기)에서
  **동시에** 흘러갈 수 있습니다.
- **왜 중요한가**: 5절의 이중 버퍼 파이프라인은 사실 "청크 i의 전송 완료 이벤트를 청크 i의 연산
  스트림이 기다리는" 구조의 확장입니다 — 이 절에서 배우는 2-스트림 의존성 패턴이 그대로 N개 청크로
  일반화됩니다.

<img src="images/figures/new_streams_events.png" width="620">

<sub>그림: 스트림 s1의 작업 완료를 이벤트로 표시하고, 스트림 s2가 그 이벤트를 기다린(wait_event)
뒤 자신의 작업을 이어가는 의존성 구조 — CPU는 두 스트림 모두 기다리지 않고 계속 진행합니다.</sub>



In [ ]:
a1=cp.random.random(20_000_000,dtype=cp.float32); a2=cp.random.random(20_000_000,dtype=cp.float32)
s1=cp.cuda.Stream(non_blocking=True); s2=cp.cuda.Stream(non_blocking=True)
e=cp.cuda.Event()
with s1:
    t=cp.sin(a1); e.record(s1)
with s2:
    s2.wait_event(e)          # s2는 e 이후 진행
    out=(t+cp.cos(a2)).sum()
s2.synchronize()
print('의존성 결합:', float(out))

<a id="4"></a>
## 4. 비동기 전송 & pinned memory

전송은 기본 블로킹입니다. CuPy 13+는 `cp.asarray(x, blocking=False)`/`cp.asnumpy(x, blocking=False)`로 비동기 전송을 합니다.
겹침 효과를 보려면 host 버퍼가 **pinned(page-locked)** 여야 합니다(`cp.cuda.alloc_pinned_memory`).

- Pinned Memory (Page-Locked Memory) 개념 정리:
"OS가 마음대로 메모리 주소를 바꾸거나 디스크로 쫓아내지 못하도록, RAM의 특정 위치에 '고정해 둔' 메모리 영역"
- 왜 Pinned Memory가 필요할까? (배경 원리)
우리가 평소에 쓰는 일반 Host 메모리(Pageable Memory)는 OS가 효율적인 관리(가상 메모리)를 위해 필요시 디스크(Swap)에 내보내거나 실제 물리 주소를 바꿉니다.
- 일반 메모리 (Pageable Memory)를 GPU로 보낼 때 일어나는 일:
CPU가 일반 RAM에 있는 데이터를 Pinned Memory(임시 버퍼)로 복사함 (추가 복사 단계 발생!)
Pinned Memory에 올라간 데이터를 DMA(Direct Memory Access)를 통해 GPU로 전송함
- Pinned Memory를 직접 만들어 보낼 때 일어나는 일:
처음부터 Pinned Memory에 데이터를 생성함.
복사 단계 없이 곧바로 DMA를 통해 GPU로 직통 전송!
- 비동기 전송(Asynchronous Transfer)과의 관계:
비동기 전송(Stream 전송)을 가능하게 만드는 필수 전제 조건이 바로 Pinned Memory입니다.
- 일반 메모리 전송 (동기식):
CPU는 GPU로 데이터 복사(Stage 과정)가 끝날 때까지 아무 일도 못 하고 대기(Blocking)
- Pinned 메모리 전송 (비동기식):
GPU 전송을 DMA(하드웨어 엔진)에 맡겨버리고, CPU는 기다리지 않고 다음 파이썬 코드를 즉시 실행합니다.
[데이터 전송]과 [GPU 연산/CPU 연산]을 동시에 진행(Overlap)할 수 있게 됩니다.
- pinned memory 사용시 주의점:
실제 물리 RAM 용량을 그대로 점유해 버리기 때문에, OS나 다른 프로세스가 사용할 수 있는 실질 메모리 양이 대폭 줄어듭니다.
Pinned Memory를 너무 과도하게 할당하면, 정작 OS 핵심 프로세스나 파이썬 메인 프로그램이 쓸 메모리가 부족해집니다.
전체 Host RAM의 50%~70% 이상을 Pinned Memory로 채우지 않도록 주의하세요.

### 조금 더 구체적으로: 왜 pinned이 없으면 "비동기"가 무의미해지는가

일반(pageable) 메모리는 물리 주소가 언제든 바뀔 수 있어서, DMA 엔진이 안전하게 참조할 수 없습니다.
그래서 CUDA 드라이버는 pageable 메모리를 전송할 때 **내부적으로 몰래 pinned 스테이징 버퍼를 만들어
한 번 더 복사**한 뒤 DMA를 겁니다 — 이 추가 복사가 끝나야 호출이 반환되므로, `blocking=False`를
지정해도 사실상 동기 전송과 다를 바 없이 느려집니다(겹칠 자원이 없으므로). CuPy는 pageable 입력에
`blocking=False`를 주면 예외 없이 **조용히 동기 전송으로 대체**되는 경우가 있어, 겹침이 안 되는데도
코드는 에러 없이 돌아간다는 점을 주의해야 합니다 — 반드시 pinned 버퍼로 실측 속도를 확인하세요.

| 항목 | Pageable 메모리 | Pinned 메모리 |
|------|------------------|----------------|
| 전송 경로 | Pageable → (드라이버 내부) Pinned 스테이징 → DMA | 바로 DMA |
| PCIe Gen4 x16 기준 typical 실효 대역폭(참고치) | 대략 절반 수준 | 이론치(약 32 GB/s 편도)에 근접 |
| `blocking=False` 실효성 | 사실상 무효(내부 복사가 끝날 때까지 대기) | 실제 비동기 — 연산과 겹침 가능 |
| 확보 비용 | `np.empty` 등 즉시 | `cp.cuda.alloc_pinned_memory` — 할당 자체가 상대적으로 느림(페이지 잠금 시스템콜) |
| 과다 사용 시 위험 | 없음 | 물리 RAM 고갈 → OS 스와핑 압박, 다른 프로세스 성능 저하 |

> 이 표의 첫 행이 바로 5절 "이중 버퍼 파이프라인"이 성립하기 위한 **전제조건**입니다. pinned 없이
> 청크를 여러 스트림에 나눠도 전송 자체가 겹치지 않으므로 속도 이득이 거의 나지 않습니다.
> `05_memory_profiling`에서 다룬 메모리 풀(`MemoryPool`)과 마찬가지로, CuPy는 pinned 메모리도
> `PinnedMemoryPool`로 재사용을 최적화할 수 있습니다(반복 할당 비용을 줄이고 싶다면 참고하세요).

In [ ]:
n=8_000_000; itemsize=np.dtype(np.float32).itemsize
pinned=cp.cuda.alloc_pinned_memory(n*itemsize)
h=np.frombuffer(pinned,dtype=np.float32,count=n); h[:]=np.random.rand(n).astype(np.float32)
s=cp.cuda.Stream(non_blocking=True)
with s:
    try: d=cp.asarray(h, blocking=False)
        # 1. 최신 CuPy인 경우: 비동기(Non-blocking)로 GPU 메모리(pinned)에 바로 복사 시도
    except TypeError: d=cp.asarray(h)
        # 2. 구버전 CuPy인 경우: 'blocking' 인자가 없으므로 TypeError 발생 -> 동기(Blocking) 전송으로 처리
    y=(d*2+1).sum()
s.synchronize(); print('pinned 비동기 전송 결과:', float(y))

<a id="5"></a>
## 5. 이중 버퍼 청크 오버랩 파이프라인
스트림의 **핵심 활용**입니다. 큰 데이터를 청크로 나눠 여러 스트림에 분산하면,
한 청크의 전송과 다른 청크의 연산이 **겹쳐** 전체 시간이 줄어듭니다. 먼저 순차 기준선과 비교합니다.

- 순차 실행 방식 (sequential)
```
* 특징: H->D 전송 중에는 GPU 코어가 놀고(Idle), GPU 연산 중에는 PCIe 전송 통로(DMA Engine)가 놉니다.
[시간 흐름 (Time) ----------------------------------------------------------------------------------------►]

청크 0 : | H->D 전송 | === GPU 연산 === | D->H 회수 |
청크 1 :                                           | H->D 전송 | === GPU 연산 === | D->H 회수 |
청크 2 :                                                                                      | H->D ...
------------------------------------------------------------------------------------------------------------
``


- 비동기 스트림 방식 (chunked(nstreams=3))
 ``
* Pinned Memory + 다중 Stream 활용
* 한 청크가 GPU 연산을 하는 동안, 다른 청크는 PCIe 통로를 통해 전송(Overlap)을 동시 진행합니다.
[시간 흐름 (Time) ----------------------------------------------------------------------------------------►]

Stream 0 (청크 0) : | H->D (0) | ====== GPU 연산 (0) ====== | D->H (0) |
Stream 1 (청크 1) :            | H->D (1) | ====== GPU 연산 (1) ====== | D->H (1) |
Stream 2 (청크 2) :                       | H->D (2) | ====== GPU 연산 (2) ====== | D->H (2) |
Stream 0 (청크 3) :                                  | H->D (3) | ====== GPU 연산 (3) ====== | D->H (3) |
------------------------------------------------------------------------------------------------------------
```

### 조금 더 구체적으로: 겹침의 이론적 한계와 청크 크기의 트레이드오프

이중 버퍼(더 일반적으로는 **N중 버퍼**) 파이프라인의 이상적인 총 실행 시간은 대략 다음과 같이
근사할 수 있습니다.

```
T_total ≈ max(T_transfer_total, T_compute_total) + T_one_chunk_latency
```

즉, 전송에 걸리는 총 시간과 연산에 걸리는 총 시간 중 **더 큰 쪽**이 전체 시간을 지배하고, 파이프라인이
채워지고 비워지는 데 걸리는 **첫/마지막 청크 한 개분의 지연(latency)** 만 추가로 붙습니다. 이것이
바로 컴퓨터 구조에서 흔히 말하는 **파이프라이닝(pipelining)** 의 일반 원리이고, CPU의 명령어
파이프라인이나 생산 라인의 컨베이어 벨트와 동일한 발상입니다 — 전송과 연산이 정확히 절반씩 걸린다면
이론상 **최대 2배**까지 겹침 이득을 볼 수 있습니다(전송이 연산보다 훨씬 크거나 작으면 이득은 그만큼
줄어듭니다).

청크 크기(`CH`)를 정할 때는 다음 트레이드오프를 고려해야 합니다.

- **청크가 너무 작으면**: 청크 개수가 늘어나 커널 런치·전송 API 호출의 **고정 오버헤드**가 누적되어
  총 시간이 오히려 늘어날 수 있습니다(`00_intro_env` 2절의 "지연시간 vs 처리량" 논의, `01_benchmark_basics`의
  손익분기점과 같은 맥락).
- **청크가 너무 크면**: 스트림 수가 적어져 겹칠 기회 자체가 줄고, 파이프라인이 채워지기 전/비워진 후의
  "가장자리 지연"이 전체 시간에서 차지하는 비중이 커집니다.
- 실무적으로는 몇 가지 청크 크기(예: 전체의 1/8, 1/16, 1/32)를 실측 비교해 sweet spot을 찾는 것이
 일반적이며, 이 노트북의 연습문제에서 스트림 수(`nstreams`)를 바꿔가며 그 감각을 직접 확인합니다.



In [ ]:
N = 64_000_000      # 전체 데이터의 개수 (6,400만 개)
CH = 4_000_000      # 한 번에 처리할 청크(Chunk)의 크기 (400만 개)
itemsize = np.dtype(np.float32).itemsize  # float32의 바이트 크기 (4바이트)
pin=cp.cuda.alloc_pinned_memory(N*itemsize)
# 고정 메모리(Pinned Memory) 할당: GPU 비동기 전송의 핵심입니다. OS가 페이지 아웃(Page-out)시키지 못하도록 메모리 상에 딱 고정시켜 둔 특수 메모리 영역을 CuPy를 통해 할당합니다. 이 영역을 쓰면 CPU와 GPU 간의 데이터 전송 속도가 훨씬 빨라집니다.
H=np.frombuffer(pin,dtype=np.float32,count=N); H[:]=np.random.rand(N).astype(np.float32)
# 호스트 배열 연결: 방금 할당한 고정 메모리(pin)를 껍데기로 삼아, NumPy 배열 H를 생성합니다. 그리고 이 배열에 임의의 랜덤 값(0~1 사이)을 채워 넣습니다. 이제 H는 고정 메모리에 매핑된 상태입니다.
def work(d): return cp.sqrt(d*d+1.0)

# 기준선(순차): 한 청크씩 전송->연산->회수 (겹침 없음)
def sequential():
    out=np.empty(N,np.float32)
# s0는 0부터 시작해 CH(4,000,000)씩 커지며 N 직전까지 반복하는 루프입니다.
    for s0 in range(0,N,CH):
        d=cp.asarray(H[s0:s0+CH]);
# 호스트 메모리의 s0부터 s0+CH 범위의 조각(16MB)을 GPU 메모리로 복사(전송)합니다. 이 작업이 끝날 때까지 CPU는 다음 줄로 넘어가지 않고 기다립니다(동기식).
        r=work(d); 
# GPU로 전송된 데이터 d를 가지고 위에서 정의한 work() 연산을 수행합니다. 연산 결과인 r 역시 GPU 메모리에 머무릅니다.
        out[s0:s0+CH]=cp.asnumpy(r)
# GPU 메모리에 있는 결과 r을 다시 호스트(CPU)의 out 배열 슬라이스 영역으로 복사하여 가져옵니다.
    return out
print_bench(bench(sequential, n_repeat=5, name='sequential'))


**연습 — 이중 버퍼 오버랩 직접 구현**: `H`를 `CH` 청크로 나눠 `nstreams`개 스트림에 분산하고,
각 청크에서 `asarray(blocking=False)`→`work`→`asnumpy(blocking=False)`로 처리해 **순차 대비 속도**를 비교하세요.

**구현 시 체크리스트**
- 청크 `i`는 `streams[i % nstreams]` 스트림에 올려, 스트림 수만큼 라운드로빈으로 순환시킵니다.
- `blocking=False` 인자는 CuPy 13 미만에서는 지원하지 않으므로 `TypeError`를 잡아 구버전 호환
  경로(동기 전송)로 넘어가게 만드는 것이 안전합니다(위 4절 셀의 패턴과 동일).
- 결과 배열 `out`에 청크별 결과를 써 넣는 시점이 실제로는 GPU 작업이 끝나기 **전**일 수 있으므로,
  모든 스트림을 다 순회한 뒤 반드시 각 스트림을 `synchronize()`해야 `out`을 안전하게 반환할 수 있습니다.
- 기대치: `H`가 pinned 메모리이고 `work`가 충분히 무거우면(여기서는 `sqrt(d*d+1)`), 2~3개
  스트림에서 순차 대비 **1.3~2배** 정도의 속도 향상을 보이는 경우가 흔합니다 — 정확한 배수는
  GPU 세대·PCIe 세대·연산 강도에 따라 달라지므로 직접 측정해서 확인하세요.

In [ ]:
def chunked(nstreams=3):
    # TODO: nstreams개 Stream(non_blocking=True) 생성
    #       각 청크를 streams[i%nstreams]에서 asarray(blocking=False)->work->asnumpy(blocking=False)
    #       모든 스트림 synchronize 후 out 반환
    raise NotImplementedError

# print_bench(bench(lambda: chunked(3), n_repeat=5, name='chunked(3)'))
# print('speedup:', round(cpu_ms(bench(sequential))/cpu_ms(bench(lambda: chunked(3))),2))

<details><summary>💡 해답 보기</summary>

```python
def chunked(nstreams=3):
    out=np.empty(N,np.float32)
    streams=[cp.cuda.Stream(non_blocking=True) for _ in range(nstreams)]
    for i,s0 in enumerate(range(0,N,CH)):
        with streams[i%nstreams]:
            try: d=cp.asarray(H[s0:s0+CH], blocking=False)
            except TypeError: d=cp.asarray(H[s0:s0+CH])
            r=work(d)
            try: out[s0:s0+CH]=cp.asnumpy(r, blocking=False)
            except TypeError: out[s0:s0+CH]=cp.asnumpy(r)
    for st in streams: st.synchronize()
    return out

for k in [1,2,4]:
    print_bench(bench(lambda k=k: chunked(k), n_repeat=5, name=f'chunked({k})'))
# 보통 2~3 스트림에서 이득이 포화됩니다.
```
</details>

<a id="6"></a>
## 6. 이벤트 세분 타이밍

전송 구간과 연산 구간을 이벤트로 분리 측정하면 어디가 병목인지 보입니다.

### 조금 더 구체적으로

`05_memory_profiling`에서는 "이 연산이 memory-bound인가 compute-bound인가"를 산술강도(arithmetic
intensity)로 미리 추정했다면, 여기서는 **실측**으로 같은 질문에 답합니다. 전송 구간(`e0`→`e1`)과
연산 구간(`e1`→`e2`)을 따로 재면 다음을 바로 판단할 수 있습니다.

- **전송 시간 ≫ 연산 시간**: 이 워크로드는 아무리 커널을 최적화해도 전체 시간이 거의 줄지 않습니다.
  이럴 때 투자해야 할 곳은 5절의 **겹침(overlap)** 이나, 애초에 전송량 자체를 줄이는 방법입니다.
- **연산 시간 ≫ 전송 시간**: 스트림 겹침으로 얻을 수 있는 이득은 제한적입니다(전송이 이미 거의
  공짜이므로). 이때는 커널 자체의 연산 효율(Day 2 단원)에 투자하는 편이 낫습니다.
- **둘이 비슷한 크기**: 겹침으로 얻는 이득이 가장 큰 구간입니다 — 5절 공식(`T_total ≈ max(전송, 연산)
  + 가장자리`)에서 `max`가 `합`보다 눈에 띄게 작아지는 지점이기 때문입니다.

이런 구간별 분해는 사실 Nsight Systems 같은 프로파일러가 타임라인에서 자동으로 보여주는 정보를
코드 안에서 직접 재현한 것입니다 — 8절의 NVTX 마킹으로 넘어가면 이 수작업 측정을 시각적 타임라인으로
확장합니다.

In [ ]:
e0=cp.cuda.Event(); e1=cp.cuda.Event(); e2=cp.cuda.Event()
e0.record()
d=cp.asarray(H[:CH])      # 전송
e1.record()
r=work(d).sum()           # 연산
e2.record(); e2.synchronize()
print(f'transfer {cp.cuda.get_elapsed_time(e0,e1):.3f} ms | compute {cp.cuda.get_elapsed_time(e1,e2):.3f} ms')

<a id="7"></a>
## 7. 스트림 수 스케일링

독립 연산을 1·2·4·8 스트림에 나눠 올리고 시간을 비교합니다. 단일 커널이 GPU를 이미 채우면 이득이 작습니다.

### 조금 더 구체적으로

스트림 수를 늘렸을 때 성능이 계속 좋아지지 않는 이유는, GPU가 겹칠 수 있는 **물리적 자원의 개수가
유한**하기 때문입니다.

- **복사 엔진(copy engine)**: 대부분의 데이터센터급 GPU는 H→D와 D→H를 동시에 처리할 수 있는 복사
  엔진을 1~2개 정도 갖습니다. 전송이 많은 워크로드라면 스트림을 아무리 늘려도 **복사 엔진 개수**
  이상으로는 전송이 동시에 진행되지 않습니다.
- **SM(연산 자원)**: 하나의 커널이 이미 GPU의 모든 SM을 점유할 만큼 크다면(=충분히 큰 배열의
  원소별 연산), 다른 스트림의 커널을 위한 유휴 SM이 남아있지 않아 사실상 순차 실행과 큰 차이가
  없습니다. 반대로 각 커널이 작아 GPU 일부만 쓴다면, 여러 개를 동시에 올려 **점유율(occupancy)**
  을 채우는 데서 이득이 납니다.
- **런치 오버헤드의 누적**: 스트림 수(=커널 개수)가 늘어날수록 CPU가 커널을 제출하는 데 드는
  고정 오버헤드도 누적되므로, 이 비용이 겹침 이득을 갉아먹기 시작하는 지점부터는 스트림을 늘려도
  성능이 정체되거나 오히려 나빠집니다.

실무적으로는 "2, 4개 정도에서 이득이 나고 8개부터는 포화되거나 미미해진다"는 패턴이 흔하며, 이
노트북 실습에서 직접 그 포화 지점을 관찰하는 것이 목표입니다. 스트림 개수를 늘리는 전략이 항상
안전한 최적화가 아니라는 점 — 즉 **"자원이 남을 때만 이득"** 이라는 원칙은 9절 CUDA Graph, 10절
멀티-GPU에서도 반복됩니다.

In [ ]:
arrs=[cp.random.random(20_000_000,dtype=cp.float32) for _ in range(8)]
def run_streams(k):
    streams=[cp.cuda.Stream(non_blocking=True) for _ in range(k)]; outs=[]
    for i,a in enumerate(arrs):
        with streams[i%k]: outs.append((cp.sin(a)+1).sum())
    for st in streams: st.synchronize()
    return outs
for k in [1,2,4,8]:
    print_bench(bench(lambda k=k: run_streams(k), n_repeat=10, name=f'{k} stream(s)'))

<a id="8"></a>
## 8. NVTX/프로파일 연계 + Power Iteration

`cupyx.profiler.time_range`로 구간을 NVTX 라벨링하면 Nsight Systems 타임라인에서 오버랩·idle을 볼 수 있습니다(단원 3 도구).
단원 3의 Power Iteration으로 **동기화 빈도**의 영향을 다시 확인합니다(잦은 `float()`=잦은 host 동기화).
`!nsys profile python ./cupy/06_8.py` 실행후, 출력파일(확장자 nsys-rep) 다운로드 하신뒤, 로컬 컴퓨터에서 nsight 프로그램을 실행해서 타임라인을 확인해보세요. `time_range`로 구간을 라벨링하세요.

### 조금 더 구체적으로

**NVTX(NVIDIA Tools Extension)** 는 코드에 사람이 읽을 수 있는 이름표를 붙여 프로파일러 타임라인에
표시해주는 마킹 시스템입니다. `time_range('이름', color_id=...)`로 감싼 구간은 Nsight Systems
타임라인에 색이 칠해진 막대로 나타나, 수백만 개의 커널 launch 사이에서 "이게 어느 논리적 단계인지"를
한눈에 찾을 수 있게 해줍니다 — 이벤트로 직접 시간을 재는 6절 방식이 **숫자**를 주는 도구라면, NVTX는
그 숫자들이 타임라인 위 **어디**에 위치하는지 시각적으로 보여주는 도구입니다.

이 절의 예제 스크립트(`06_8.py`)는 Power Iteration을 두 가지 방식으로 실행합니다.

- **매 스텝 동기화**(`check_every=1`): 매 반복마다 `float(nrm)`을 호출해 GPU 스칼라를 CPU로
  가져옵니다. `00_intro_env`에서 배웠듯 `float()`/`.item()` 호출은 **강제로 host 동기화**를
  일으키므로, 300번의 반복마다 300번씩 "GPU가 끝날 때까지 CPU가 기다리는" 지점이 생깁니다.
- **50스텝마다 동기화**(`check_every=50`): 같은 총 연산량이지만 동기화 횟수가 1/50로 줄어,
  GPU가 커널들을 **끊김 없이 연속으로** 제출받아 실행할 수 있습니다.

Nsight Systems 타임라인에서 두 구간(`pi_check1` vs `pi_check50`)을 비교하면, 전자는 GPU 실행
막대 사이사이에 **작은 공백(bubble)** 이 촘촘히 보이고(동기화 대기), 후자는 막대가 훨씬 촘촘하게
이어져 있는 것을 확인할 수 있습니다 — 같은 알고리즘, 같은 총 연산량이라도 **동기화 빈도** 하나로
GPU 활용률이 크게 달라진다는 사실을 시각적으로 확인하는 것이 이 실습의 핵심입니다.

In [ ]:
%%writefile 06_8.py
import cupy as cp
from cupyx.profiler import time_range, benchmark
from cupy.cuda import profiler

# 1. 테스트용 행렬 준비
M = cp.random.random((1500, 1500), dtype=cp.float32)
A = (M + M.T) / 2

def power_iter_sync(A, iters=300, check_every=1):
    xp = cp.get_array_module(A)
    x = xp.ones(A.shape[0], dtype=A.dtype)
    for i in range(iters):
        y = A @ x
        nrm = xp.linalg.norm(y)
        x = y / nrm
        if i % check_every == 0:
            _ = float(nrm)   # host 동기화 (CPU-GPU 병목 유발)
    return x

# 프로파일링 명시적 시작
profiler.start()

# [구간 1] 매 스텝 동기화
with time_range('pi_check1', color_id=0):
    # bench 실행
    res1 = benchmark(lambda: power_iter_sync(A, check_every=1), n_repeat=5, name='sync 매 스텝')
    # ★ 핵심: NVTX 구간(time_range)이 닫히기 전에 GPU 연산이 모두 끝날 때까지 대기
    cp.cuda.Device().synchronize()

print(res1)

# [구간 2] 50스텝마다 동기화
with time_range('pi_check50', color_id=1):
    res2 = benchmark(lambda: power_iter_sync(A, check_every=50), n_repeat=5, name='sync 50스텝마다')
    # ★ 핵심: NVTX 구간이 닫히기 전 GPU 동기화
    cp.cuda.Device().synchronize()

print(res2)

# 프로파일링 종료 및 버퍼 Flush
profiler.stop()

<details><summary>(선택) Nsight Systems 프로파일링 워크플로</summary>

이 셀은 실제 GPU 서버에서 `nsys` CLI로 프로파일 파일(`.nsys-rep`)을 만들고, 그 결과물을 로컬
PC로 내려받아 Nsight Systems GUI(또는 Perfetto 웹뷰어)로 여는 표준 워크플로를 보여줍니다.
`--capture-range=cudaProfilerApi`와 `--capture-range-end=stop` 옵션은 스크립트 전체가 아니라
`cupy.cuda.profiler.start()`/`stop()`로 감싼 구간만 골라서 캡처하도록 지정하는 것으로, 불필요한
초기화 구간까지 캡처해 파일이 커지고 분석이 번거로워지는 것을 막아줍니다(8절 스크립트의
`profiler.start()`/`profiler.stop()` 호출과 정확히 대응됩니다).

```python
# %%writefile pi.py  로 스크립트 저장 후 터미널에서:
# nsys profile --capture-range=cudaProfilerApi --capture-range-end=stop -o pi python pi.py
# 생성된 pi.nsys-rep 를 Nsight Systems GUI / Perfetto 에서 열어 타임라인 확인
```
`with cupyx.profiler.profile():` 블록 안에서만 캡처되며, `time_range` 라벨이 타임라인에 표시됩니다.

> GUI에서 타임라인을 열면 CUDA HW 트랙에 커널 실행/메모리 복사 막대가, NVTX 트랙에 `time_range`로
> 붙인 이름표가 겹쳐 표시됩니다. 두 트랙을 함께 보면 "이 NVTX 구간 동안 GPU가 얼마나 바빴는가"를
> 바로 읽을 수 있어, 6~7절에서 숫자로 확인한 겹침·병목을 눈으로 다시 검증하는 셈이 됩니다.
</details>

<a id="9"></a>
## 9. (심화) CUDA Graph 캡처

📖 [`cupy.cuda.Graph`](https://docs.cupy.dev/en/stable/reference/generated/cupy.cuda.Graph.html) — 반복되는 스트림 작업 시퀀스를 **그래프로 캡처**해 한 번에 실행하면 **커널 런치 오버헤드**가 줄어듭니다.
`stream.begin_capture()` … `g = stream.end_capture()` … `g.launch()`. 캡처 중에는 **host 동기 전송 금지**입니다.

- 일반적으로 CuPy(또는 CUDA) 코드를 실행할 때, CPU는 GPU에게 "A 커널(함수)을 실행해", 그다음 "B 커널을 실행해"라고 매번 명령을 내립니다. 이 명령을 내리는 시간(CPU-GPU 통신 시간)을 커널 런치 오버헤드(Kernel Launch Overhead)라고 합니다.
- 작업 단위가 크다면 이 오버헤드가 무시할 만하지만, 연산이 매우 짧고 자잘한 커널을 수천~수만 번 반복해서 호출해야 한다면(예: 딥러닝 학습 루프, 반복적인 물리 시뮬레이션) 연산 시간보다 명령을 내리는 대기 시간이 더 길어지는 병목 현상이 발생합니다.
- CUDA Graph는 이 문제를 해결합니다. 일련의 GPU 작업(메모리 복사, 연산 등)을 실행하는 대신 "기록(Capture)"하여 하나의 거대한 작업 흐름도(Graph)로 묶어둡니다. 이후에는 CPU가 "저장된 그래프 실행해"라고 단 한 번만 명령하면 되므로 런치 오버헤드가 획기적으로 줄어듭니다.
- 그래프를 캡처하는 구간(begin_capture ~ end_capture)은 연산을 실행하는 것이 아니라 레시피를 적는 시간입니다. 따라서 GPU가 연산을 끝낼 때까지 기다렸다가 결과값을 CPU로 가져오는(Host Synchronization) 코드가 캡처 구간 안에 있으면 에러가 발생하거나 캡처가 중단됩니다.
   * 캡처 중 절대 하면 안 되는 행동:
     * .get() 또는 cp.asnumpy() 호출: GPU 배열을 CPU 넘파이 배열로 변환하는 행위는 동기화를 강제합니다.
     * 조건문(if)에 GPU 값 사용: if a.sum() > 0: 와 같은 코드를 쓰면, 조건을 평가하기 위해 CPU가 GPU의 sum() 결과를 기다려야 하므로 동기화가 발생합니다.
     * stream.synchronize() 호출: 명시적인 동기화 명령이므로 당연히 금지됩니다.
- 모든 코드에 그래프를 적용할 필요는 없습니다. 다음과 같은 상황에서 성능 향상을 보입니다.
   * 반복 횟수가 매우 많은 루프: AI 모델의 에폭(Epoch) 반복, MCMC 샘플링 등 동일한 구조의 연산이 수백 번 이상 반복될 때.
   * 커널 크기가 작을 때: 개별 행렬 곱셈이나 덧셈 연산 자체는 1ms 이내로 끝나는데, 이를 연속해서 호출해야 할 때.
   * 제어 흐름(Control Flow)이 고정되어 있을 때: if-else 조건에 따라 연산 그래프 구조 자체가 매번 바뀌는 동적 모델(Dynamic Graph)에는 적용하기 어렵습니다.

### 조금 더 구체적으로: 오버헤드의 크기와 캡처의 제약

커널 런치 하나의 CPU측 오버헤드는 대략 **수 마이크로초(µs) 단위**로 알려져 있습니다. 커널 자체의
실행 시간이 그와 비슷하거나 더 짧다면(예: 원소 수가 적은 elementwise 연산), 전체 시간의 대부분이
"일을 시키는 시간"이지 "일하는 시간"이 아니게 됩니다. CUDA Graph는 이런 경우 반복되는 시퀀스를
**한 번의 그래프 런치**로 묶어, 시퀀스 안 커널 개수와 무관하게 CPU 오버헤드를 사실상 상수로
만들어 줍니다 — 이 노트북의 `many_small` 예제(50개의 작은 `cp.add` 반복)가 정확히 이 상황을 재현합니다.

캡처를 실무에 적용할 때 흔히 놓치는 제약도 함께 알아둘 필요가 있습니다.

- **입력 포인터/shape가 고정됩니다**: 캡처된 그래프는 캡처 당시 사용된 GPU 메모리 주소를 그대로
  기록합니다. 따라서 `g.launch()`를 반복 호출할 때는 **같은 배열 객체(같은 메모리 주소)** 에 새
  값을 덮어써야 하며, 매번 새 배열을 만들어 그래프에 넘기는 방식은 동작하지 않습니다.
- **캡처는 기본이 아닌 스트림에서** 이뤄져야 합니다(`stream.begin_capture()`). 기본 스트림은
  다른 스트림과 암묵적으로 동기화되는 특수한 성격 때문에 캡처와 궁합이 좋지 않습니다(1절 참고).
- CuPy/CUDA 버전과 드라이버에 따라 그래프 캡처 지원 범위가 다를 수 있어, 이 노트북의 코드도
  `AttributeError`/`RuntimeError`를 잡아 지원하지 않는 환경에서는 조용히 건너뛰도록 방어적으로
  작성되어 있습니다.

In [ ]:
# 작은 커널을 여러 번 실행하는 반복 시퀀스 (런치 오버헤드 지배)
x=cp.random.random(1_000_000,dtype=cp.float32)
def many_small():
    for _ in range(50): cp.add(x,1.0,out=x)

s=cp.cuda.Stream(non_blocking=True)
try:
    with s:
        s.begin_capture()
        for _ in range(50): cp.add(x,1.0,out=x)
        g=s.end_capture()
    print_bench(bench(many_small, n_repeat=50, name='반복 런치'))
    print_bench(bench(lambda: g.launch(), n_repeat=50, name='CUDA Graph'))
except (AttributeError, RuntimeError) as ex:
    print('이 환경에서는 그래프 캡처를 건너뜁니다:', ex)

<a id="10"></a>
## 10. (심화) 멀티-GPU

GPU가 여러 개면 `with cp.cuda.Device(i):` 로 장치를 전환합니다. 연산 입력은 같은 장치에 있어야 합니다.

### 조금 더 구체적으로

CuPy에서 멀티-GPU는 (아직) 자동 분산이 아니라 **사용자가 명시적으로 장치를 오가며 배열을 배치**하는
모델입니다. `with cp.cuda.Device(i):` 블록 안에서 만든 배열은 그 GPU의 메모리에 할당되고, CuPy는
서로 다른 장치에 있는 배열끼리 직접 연산(`a0 + a1`처럼 device 0과 device 1의 배열을 섞어 쓰는 것)을
허용하지 않습니다 — 시도하면 에러가 나거나(버전에 따라) 암묵적으로 잘못된 결과를 낼 수 있으므로,
연산 전에는 항상 **같은 장치로 옮긴** 뒤 계산해야 합니다.

- **디바이스 간 이동**: 장치 0의 배열을 장치 1에서 쓰려면 한 번 host를 거치거나(`cp.asnumpy` →
  `cp.asarray`), GPU끼리 직접 통신 가능한 경로(**P2P/NVLink**, 지원되는 하드웨어에서 `enable_peer_access`
  류의 API)를 이용해야 합니다. P2P가 가능하면 host를 거치지 않아 훨씬 빠릅니다.
- **왜 이 절이 "맛보기"인가**: 진짜 멀티-GPU 워크로드(여러 GPU에 데이터를 나눠 학습·시뮬레이션하는
  분산 처리)는 보통 **NCCL**(NVIDIA Collective Communications Library) 같은 통신 라이브러리로
  all-reduce/broadcast 같은 집합 연산을 최적화합니다. 이 코스는 CuPy의 단일 GPU 가속에 집중하므로,
  여기서는 "장치 전환이 어떻게 동작하는지"의 개념만 확인하고 넘어갑니다.
- 이 실습 환경이 단일 GPU라면 아래 셀은 그 사실을 확인하고 건너뛰는 것이 정상 동작입니다 — 멀티-GPU
  코드 경로 자체는 그대로 읽어보되, 실제 겹침·통신 최적화까지 다루지는 않는다는 점을 염두에 두세요.

In [ ]:
ngpu=cp.cuda.runtime.getDeviceCount()
print('GPU 개수:', ngpu)
if ngpu>=2:
    with cp.cuda.Device(0): a0=cp.random.random(1_000_000,dtype=cp.float32); s0=float(a0.sum())
    with cp.cuda.Device(1): a1=cp.random.random(1_000_000,dtype=cp.float32); s1=float(a1.sum())
    print('device0 sum',s0,'| device1 sum',s1)
else:
    print('단일 GPU 환경 — 멀티-GPU 예제는 건너뜁니다.')

<a id="11"></a>
## 11. 체크포인트

- [ ] 스트림(순서)과 다른 스트림 간 오버랩, 이벤트 동기화를 이해했다
- [ ] **이중 버퍼 청크 파이프라인**으로 전송-연산을 겹쳐 속도를 높였다
- [ ] 이벤트로 전송 vs 연산 구간을 분리 측정했다
- [ ] 스트림 수 스케일링과 동기화 빈도의 영향을 확인했다
- [ ] (심화) CUDA Graph 캡처/멀티-GPU를 시도했다

**Day 1 완료!** 다음은 **`day1_capstone`**(통합 실습) → Day 2 **`07_cupy_kernels`**.